In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
#Data Gathering
df = pd.read_csv('C:\\Users\\Dell\\fake-news-detector\\data\\english\\processed\\english_combined.csv')
df.head()

,title,text,subject,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,1


In [4]:
 df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   label    44898 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.4+ MB


In [5]:
df['label'].value_counts()


label
0    23481
1    21417
Name: count, dtype: int64

In [6]:
df.shape

(44898, 4)

In [7]:
df.isna().sum()

title      0
text       0
subject    0
label      0
dtype: int64

In [8]:
df = df.dropna()

In [9]:
df.reset_index( inplace=True)
df.head()


,index,title,text,subject,label
0,0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,0
1,1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,1
2,2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,1
3,3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,0
4,4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,1


In [13]:
df = df.drop([ 'text', 'subject'], axis=1)
(df.head())

,index,title,label
0,0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,1,Trump drops Steve Bannon from National Securit...,1
2,2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,4,Donald Trump heads for Scotland to reopen a go...,1


In [14]:
df.head()

,index,title,label
0,0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,1,Trump drops Steve Bannon from National Securit...,1
2,2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,4,Donald Trump heads for Scotland to reopen a go...,1


In [15]:
sample_data = "the quick brown fox jumps over the lazy dog"
sample_data = sample_data.split()
sample_data

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

In [16]:
sample_data = [data.lower() for data in sample_data]
sample_data

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

In [17]:
stopwords = stopwords.words('english')
print(stopwords[0:10])
print(len(stopwords))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
198


In [18]:
sample_data = [data for data in sample_data if data not in stopwords]
print(sample_data)
len(sample_data)

['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']


6

In [19]:
#stemming
ps = PorterStemmer()
sample_data_stemming = [ps.stem(data) for data in sample_data]
print(sample_data_stemming)

['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']


In [22]:
#lemmatization
lm = WordNetLemmatizer()
sample_data_lemma = [lm.lemmatize(data) for data in sample_data]
print(sample_data_lemma)


['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']


In [24]:
lm = WordNetLemmatizer()
corpus = []
for i in range (len(df)):
    review = re.sub('^a-zA-Z0-9',' ', df['title'][i])
    review = review.lower()
    review = review.split()
    review = [lm.lemmatize(x) for x in review if x not in stopwords]
    review = " ".join(review)
    corpus.append(review)

In [25]:
df['title'][0]

'Ben Stein Calls Out 9th Circuit Court: Committed a ‘Coup d’état’ Against the Constitution'

In [26]:
corpus[0]

'ben stein call 9th circuit court: committed ‘coup d’état’ constitution'

In [ ]:
#Vectorization
tf = TfidfVectorizer()
x = tf.fit_transform(corpus).toarray()
x


In [ ]:
y = df['label']
y.head()

0    0
1    1
2    1
3    0
4    1
Name: label, dtype: int64

In [ ]:
#Data splitting into train and test
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state = 10, stratify = y )


In [ ]:

len(x_train),len(y_train)

(31428, 13470, 31428, 13470)

In [ ]:
len(x_test), len(y_test)

In [31]:
rf = RandomForestClassifier()
rf.fit(x_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [32]:
#model evaluation
y_pred = rf.predict(x_test)
accuracy_score_ = accuracy_score(y_test, y_pred)
accuracy_score_

0.944988864142539

In [42]:
class Evaluation:
    
    def __init__(self,model,x_train,x_test,y_train,y_test):
        self.model = model
        self.x_train = x_train
        self.x_test = x_test
        self.y_train = y_train
        self.y_test = y_test
        
    def train_evaluation(self):
        y_pred_train = self.model.predict(self.x_train)
        
        acc_scr_train = accuracy_score(self.y_train,y_pred_train)
        print("Accuracy Score On Training Data Set :",acc_scr_train)
        print()
        
        con_mat_train = confusion_matrix(self.y_train,y_pred_train)
        print("Confusion Matrix On Training Data Set :\n",con_mat_train)
        print()
        
        class_rep_train = classification_report(self.y_train,y_pred_train)
        print("Classification Report On Training Data Set :\n",class_rep_train)
        
        
    def test_evaluation(self):
        y_pred_test = self.model.predict(self.x_test)
        
        acc_scr_test = accuracy_score(self.y_test,y_pred_test)
        print("Accuracy Score On Testing Data Set :",acc_scr_test)
        print()
        
        con_mat_test = confusion_matrix(self.y_test,y_pred_test)
        print("Confusion Matrix On Testing Data Set :\n",con_mat_test)
        print()
        
        class_rep_test = classification_report(self.y_test,y_pred_test)
        print("Classification Report On Testing Data Set :\n",class_rep_test)

        



In [43]:
#checking the accuracy on training dataset
Evaluation(rf,x_train, x_test, y_train, y_test).train_evaluation()


Accuracy Score On Training Data Set : 1.0

Confusion Matrix On Training Data Set :
 [[16436     0]
 [    0 14992]]

Classification Report On Training Data Set :
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     16436
           1       1.00      1.00      1.00     14992

    accuracy                           1.00     31428
   macro avg       1.00      1.00      1.00     31428
weighted avg       1.00      1.00      1.00     31428



In [44]:
#checking the accuracy on testing dataset
Evaluation(rf,x_train, x_test, y_train, y_test).test_evaluation()

Accuracy Score On Testing Data Set : 0.944988864142539

Confusion Matrix On Testing Data Set :
 [[6588  457]
 [ 284 6141]]

Classification Report On Testing Data Set :
               precision    recall  f1-score   support

           0       0.96      0.94      0.95      7045
           1       0.93      0.96      0.94      6425

    accuracy                           0.94     13470
   macro avg       0.94      0.95      0.94     13470
weighted avg       0.95      0.94      0.95     13470



In [53]:
#prediction pipeline
class Preprocessing:
    def __init__(self, data):
        self.data = data 

    def test_preprocessing_user(self):
        lm = WordNetLemmatizer()
        pred_data = [self.data]

        preprocess_data = []

        for data in pred_data:
            review = re.sub('^a-zA-Z0-9',' ', data)
            review = review.lower()
            review = review.split()
            review = [lm.lemmatize(x) for x in review if x not in stopwords]
            review = " ".join(review)
            preprocess_data.append(review)
        return preprocess_data

In [54]:
df['title'][5]

' Paul Ryan Responds To Dem’s Sit-In On Gun Control In The Most DISGUSTING Way (VIDEO)'

In [55]:
data =' Paul Ryan Responds To Dem’s Sit-In On Gun Control In The Most DISGUSTING Way (VIDEO)'
Preprocessing(data).test_preprocessing_user()

['paul ryan responds dem’s sit-in gun control disgusting way (video)']

In [ ]:
class Prediction:
    def __init__(self, pred_data, model):
        self.pred_data = pred_data
        self.model = model

    def prediction_model(self):
        preprocess_data = Preprocessing(self.pred_data).test_preprocessing_user()
        data = tf.transform(preprocess_data)
        prediction = self.model.predict(data)
        
        if prediction[0] == 0:
            return "The News is Fake "
        else:
            return "The News is Real"


In [57]:
data = 'MEET THE TRUMPS: In Case You Missed It….Here’s Barbara Walters Exclusive Interview With The Trumps [VIDEO]'
Prediction(data, rf).prediction_model()

'The News is Fake '

In [58]:
df['title'][15000]


'Syrian army, allies step up attacks on last militant bastion in Deir al-Zor city'

In [59]:
user_data = 'Headless torso could belong to submarine journalist: Danish police'
Prediction(user_data, rf).prediction_model()

'The News is Real'

In [60]:
data = 'Ben Stein Calls Out 9th Circuit Court: Committed to the Constitution'
Prediction(data, rf).prediction_model()

'The News is Real'